# ToysAD-8K Frozen Development Pipeline

**Notebook file:** `08e-toysad8k-frozen-development-pipeline.ipynb`

This notebook applies the frozen automatic route to the complete development role. It processes the 100-scene screen and the remaining 600 development scenes with LocateAnything-3B, SAM 2.1 Hiera-large, DINOv2-small, B0 and P1. The existing 300-scene confirmation result is reused only when every split, model, configuration, preprocessing and schema hash matches.

Predictions are written and locked before any development evaluator resource can be opened. Calibration and internal-lock scenes remain sealed.

## Saved-output evidence copy

This notebook records the completed run for repository review. Its code and markdown match the clean runnable notebook. Private scene visuals and identifier-bearing output were removed. Safe aggregate tables and charts were retained.

Run stage: `08E Kaggle Version 5`  
Clean notebook SHA-256: `878bca2c5baae0b26507a9e58ace64400d5aa7e68fd833aa24f0a7a6f1e4e015`  
Downloaded saved-output SHA-256: `f0713957e2db7da18e77468b4d9bab060f806803f966e6933cf85fa02d0a9540`


## Acceptance contract

- The input contains the fixed 100-scene screen and 600-scene development remainder, with all 20 views.
- LocateAnything uses the frozen prompt and `area_nms_075_cap8` proposal policy.
- SAM 2.1 and DINOv2-small use immutable model revisions and fixed preprocessing.
- Every stage writes content-addressed shards that an identical execution can resume.
- Zero-proposal, empty-mask and feature failures remain explicit terminal outcomes.
- Global DINOv2 embeddings are stored as hashed float16 arrays with an object-level index.
- The model-safe boundary contains `prediction_manifest.jsonl`, `global_embeddings.npy`, `embedding_index.jsonl` and `prediction_lock.json`.
- The prediction manifest and embeddings are locked before evaluator access.
- The 300-scene confirmation result is combined only after a complete identity check.
- The 6,000-view DS2 representation backfill has a separate bounded recovery bundle bound to its verified proposal and SAM records.
- Prediction figures contain no protected labels. Evaluator correspondence, when inspected, is marked reference-only and is not a deployment claim.

In [1]:
from __future__ import annotations

from hashlib import sha256
import json
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image as NotebookImage, display

sys.dont_write_bytecode = True

NOTEBOOK_NAME = "08e-toysad8k-frozen-development-pipeline.ipynb"
EXECUTION_ENVIRONMENT = "kaggle" if Path("/kaggle/input").is_dir() else "local"
RUN_PHASE = os.getenv("VPI_RUN_PHASE", "full").strip().lower()
if RUN_PHASE not in {"incremental", "full"}:
    raise ValueError("VPI_RUN_PHASE must be incremental or full")
MAXIMUM_NEW_SHARDS = int(os.getenv("VPI_MAXIMUM_NEW_SHARDS", "10"))
if MAXIMUM_NEW_SHARDS < 1:
    raise ValueError("VPI_MAXIMUM_NEW_SHARDS must be positive")
DS2_BACKFILL_PHASE = os.getenv("VPI_DS2_BACKFILL_PHASE", "full").strip().lower()
if DS2_BACKFILL_PHASE not in {"incremental", "full"}:
    raise ValueError("VPI_DS2_BACKFILL_PHASE must be incremental or full")
OUTPUT_ROOT = Path(os.getenv(
    "VPI_OUTPUT_ROOT",
    "/kaggle/working/toysad8k_frozen_development_pipeline"
    if EXECUTION_ENVIRONMENT == "kaggle"
    else ".local/runs/toysad8k-frozen-development-pipeline",
)).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RECOVERY_OUTPUT_ROOT = Path(os.getenv(
    "VPI_RECOVERY_OUTPUT_ROOT",
    "/kaggle/working/notebook08e_recovery"
    if EXECUTION_ENVIRONMENT == "kaggle"
    else ".local/runs/notebook08e_recovery",
)).expanduser().resolve()
RECOVERY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DS2_RECOVERY_OUTPUT_ROOT = Path(os.getenv(
    "VPI_DS2_RECOVERY_OUTPUT_ROOT",
    "/kaggle/working/notebook08e_ds2_backfill_recovery"
    if EXECUTION_ENVIRONMENT == "kaggle"
    else ".local/runs/notebook08e_ds2_backfill_recovery",
)).expanduser().resolve()
DS2_RECOVERY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [2]:
def kaggle_roots() -> list[Path]:
    """Return bounded roots for both Kaggle input mount layouts."""

    input_root = Path("/kaggle/input")
    if not input_root.is_dir():
        return []
    roots: list[Path] = []
    for pattern in (
        "*",
        "datasets/*/*",
        "datasets/*/*/versions/*",
        "notebooks/*/*",
        "notebooks/*/*/versions/*",
    ):
        roots.extend(path for path in input_root.glob(pattern) if path.is_dir())
    return sorted(set(roots), key=lambda path: path.as_posix())


def find_code_root() -> Path:
    configured = os.getenv("VPI_CODE_ROOT")
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend((Path.cwd(), *Path.cwd().parents, *kaggle_roots()))
    for candidate in candidates:
        if (candidate / "scripts/run_frozen_development_pipeline.py").is_file():
            return candidate.resolve()
        if (candidate / "code/scripts/run_frozen_development_pipeline.py").is_file():
            return (candidate / "code").resolve()
    mounted_names = [path.as_posix() for path in kaggle_roots()]
    raise FileNotFoundError(
        f"Could not locate the Visual Peer Inspector code snapshot. Mounted inputs: {mounted_names}"
    )


def find_model_input(cohort_id: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else []
    candidates.extend(kaggle_roots())
    for mounted in kaggle_roots():
        candidates.extend(path for path in mounted.iterdir() if path.is_dir())
    for candidate in candidates:
        record_path = candidate / "package_record.json"
        if not record_path.is_file():
            continue
        try:
            record = json.loads(record_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if (
            record.get("cohort_id") == cohort_id
            and (candidate / "neutral_corpus_manifest.jsonl").is_file()
            and (candidate / "media").is_dir()
        ):
            return candidate.resolve()
    raise FileNotFoundError(f"Could not locate model-safe cohort {cohort_id}")


def find_package_root(package_kind: str, variable: str) -> Path:
    configured = os.getenv(variable)
    candidates = [Path(configured).expanduser()] if configured else []
    for root in kaggle_roots():
        candidates.extend(path.parent for path in root.rglob("package_record.json"))
    matches = []
    for candidate in candidates:
        record_path = candidate / "package_record.json"
        if not record_path.is_file():
            continue
        try:
            record = json.loads(record_path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if record.get("package_kind") == package_kind:
            matches.append(candidate.resolve())
    unique = sorted(set(matches), key=lambda path: path.as_posix())
    if not unique:
        raise FileNotFoundError(f"Could not locate private package {package_kind}")
    digests = {
        sha256((path / "package_record.json").read_bytes()).hexdigest()
        for path in unique
    }
    if len(digests) != 1:
        raise ValueError(f"Mounted copies of {package_kind} differ")
    return unique[0]


def find_recovery_pair(
    stem: str = "notebook08e",
    environment_name: str = "VPI_RESUME_ROOT",
) -> tuple[Path, Path] | None:
    configured = os.getenv(environment_name)
    sources = [Path(configured).expanduser()] if configured else kaggle_roots()
    bundles = sorted({
        path.resolve()
        for source in sources
        for name in (f"{stem}-recovery.zip", f"{stem}-recovery.zip.bin")
        for path in source.rglob(name)
        if path.is_file()
    })
    manifests = sorted({
        path.resolve()
        for source in sources
        for path in source.rglob(f"{stem}-recovery-manifest.json")
        if path.is_file()
    })
    if not bundles and not manifests:
        return None
    if len(bundles) != 1 or len(manifests) != 1:
        raise ValueError(
            f"{environment_name}: expected one {stem} recovery pair; "
            f"found {len(bundles)} bundles and {len(manifests)} manifests. "
            "Upload the archive as .zip.bin so Kaggle preserves its bytes."
        )
    return bundles[0], manifests[0]


def code_environment() -> dict[str, str]:
    environment = os.environ.copy()
    source_root = str(CODE_ROOT / "src")
    current = environment.get("PYTHONPATH")
    environment["PYTHONPATH"] = source_root if not current else source_root + os.pathsep + current
    return environment


def read_secret(name: str) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)


def finish_clearml(safe_root: Path, task_name: str, receipt: Path, result_tag: str) -> dict:
    """Send aggregate evidence through a bounded subprocess after inference."""

    mode = "online" if EXECUTION_ENVIRONMENT == "kaggle" else "disabled"
    environment = code_environment()
    if mode == "online":
        access = read_secret("CLEARML_API_ACCESS_KEY")
        secret = read_secret("CLEARML_API_SECRET_KEY")
        if not access or not secret:
            mode = "disabled"
        else:
            environment.update({
                "CLEARML_API_ACCESS_KEY": access,
                "CLEARML_API_SECRET_KEY": secret,
                "CLEARML_API_HOST": "https://api.clear.ml",
                "CLEARML_WEB_HOST": "https://app.clear.ml",
                "CLEARML_FILES_HOST": "https://files.clear.ml",
            })
    command = [
        sys.executable,
        str(CODE_ROOT / "scripts/finalize_ds2_clearml.py"),
        "--safe-root", str(safe_root),
        "--receipt", str(receipt),
        "--stage", "frozen_development_pipeline",
        "--project", "Visual Peer Inspector/05 Development Freeze",
        "--task-name", task_name,
        "--tag", "notebook-08e",
        "--tag", "development-freeze",
        "--tag", "kaggle",
        "--tag", result_tag,
        "--mode", mode,
        "--wall-time-seconds", "180",
    ]
    try:
        subprocess.run(command, check=True, timeout=240, env=environment)
    except Exception as error:
        receipt.parent.mkdir(parents=True, exist_ok=True)
        receipt.write_text(json.dumps({
            "schema_version": 1,
            "status": "failed",
            "error_type": type(error).__name__,
            "task_id": None,
        }, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return json.loads(receipt.read_text(encoding="utf-8"))

## Resolve the frozen inputs

In [3]:
CODE_ROOT = find_code_root()
sys.path.insert(0, str(CODE_ROOT / "src"))
from visual_peer_inspector.notebook_provenance import verify_notebook_code_snapshot
from visual_peer_inspector.data.ds2_reuse_input import verify_ds2_reuse_input

CODE_PROVENANCE = verify_notebook_code_snapshot(
    CODE_ROOT,
    "notebooks/development-freeze/08e-toysad8k-frozen-development-pipeline.ipynb",
)

DS1_ROOT = find_model_input("toysad8k-development-screen-100-v1", "VPI_DS1_MODEL_INPUT_ROOT")
REMAINDER_ROOT = find_model_input("toysad8k-development-remainder-600-v1", "VPI_REMAINDER_MODEL_INPUT_ROOT")
DS2_RGB_ROOT = find_model_input("toysad8k-development-confirm-300-v1", "VPI_DS2_RGB_ROOT")
DS2_REUSE_ROOT = find_package_root("ds2-frozen-route-reuse", "VPI_DS2_REUSE_ROOT")
DS2_REUSE_RECORD = verify_ds2_reuse_input(DS2_REUSE_ROOT)
SELECTED_PROMPT = DS2_REUSE_ROOT / "selected_prompt.json"

display({
    "execution environment": EXECUTION_ENVIRONMENT,
    "run phase": RUN_PHASE,
    "maximum new shards per stage": MAXIMUM_NEW_SHARDS,
    "previous recovery attached": find_recovery_pair() is not None,
    "screen scenes": 100,
    "development remainder scenes": 600,
    "views per scene": 20,
    "confirmation backfill scenes": 300,
    "confirmation reuse boundary": "LocateAnything and SAM identities must verify",
    "confirmation reuse package": DS2_REUSE_RECORD["package_kind"],
    "confirmation reuse hash": DS2_REUSE_RECORD["content_checksums_sha256"],
})

{'execution environment': 'kaggle',
 'run phase': 'full',
 'maximum new shards per stage': 10,
 'previous recovery attached': True,
 'screen scenes': 100,
 'development remainder scenes': 600,
 'views per scene': 20,
 'confirmation backfill scenes': 300,
 'confirmation reuse boundary': 'LocateAnything and SAM identities must verify',
 'confirmation reuse package': 'ds2-frozen-route-reuse',
 'confirmation reuse hash': 'e253fc5bd1d8ff1e9b22dcc9205951e88b9b977b452dcc12eafca4e7248ec43e'}

## Verify the frozen identity and GPU runtime

In [4]:
from visual_peer_inspector.workflows.frozen_development_pipeline import verify_prediction_lock
from visual_peer_inspector.models.sam2 import Sam2Configuration
from visual_peer_inspector.data.model_safe_package import file_sha256

checkpoint_path = None

install_environment = code_environment()
install_environment["SAM2_BUILD_CUDA"] = "0"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements/confirmation-lock.txt")],
    check=True,
    timeout=900,
    env=install_environment,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4",
    ],
    check=True,
    timeout=600,
    env=install_environment,
)

import torch
from huggingface_hub import hf_hub_download

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 08E requires a Kaggle GPU")
sam_configuration = Sam2Configuration()
checkpoint_path = Path(hf_hub_download(
    repo_id=sam_configuration.model_repository,
    filename=sam_configuration.checkpoint_filename,
    revision=sam_configuration.model_revision,
    cache_dir="/tmp/vpi-sam21-model-cache",
))
if (
    checkpoint_path.stat().st_size != sam_configuration.checkpoint_size_bytes
    or file_sha256(checkpoint_path) != sam_configuration.checkpoint_sha256
):
    raise RuntimeError("The pinned SAM 2.1 checkpoint identity differs")

display({
    "GPU": torch.cuda.get_device_name(0),
    "GPU count": torch.cuda.device_count(),
    "LocateAnything": "nvidia/LocateAnything-3B at c32291ca5e996f5a7a485845b4f57a233936bba0",
    "SAM": "facebook/sam2.1-hiera-large at 665f8e2ad61cf5f53d65644ff27c8ee525124610",
    "DINO": "facebook/dinov2-small at ed25f3a31f01632728cabb09d1542f84ab7b0056",
    "proposal policy": "area_nms_075_cap8",
})

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.9 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


sam2.1_hiera_large.pt:   0%|          | 0.00/898M [00:00<?, ?B/s]

{'GPU': 'Tesla T4',
 'GPU count': 2,
 'LocateAnything': 'nvidia/LocateAnything-3B at c32291ca5e996f5a7a485845b4f57a233936bba0',
 'SAM': 'facebook/sam2.1-hiera-large at 665f8e2ad61cf5f53d65644ff27c8ee525124610',
 'DINO': 'facebook/dinov2-small at ed25f3a31f01632728cabb09d1542f84ab7b0056',
 'proposal policy': 'area_nms_075_cap8'}

## Run the frozen automatic route

The three neural models load sequentially so that their weights do not compete for GPU memory. Each stage is split into deterministic shards. A completed shard is reused only when its input hash, output hash, embedding hashes and full pipeline identity still match. Incremental runs stop after a bounded number of new shards per stage and write a verified recovery ZIP. Attach that private recovery output to the next Kaggle version through `VPI_RESUME_ROOT`. ClearML is not active during model loading or inference.

In [5]:
command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_frozen_development_pipeline.py"),
    "run-development",
    "--ds1-model-input-root", str(DS1_ROOT),
    "--remainder-model-input-root", str(REMAINDER_ROOT),
    "--selected-prompt", str(SELECTED_PROMPT),
    "--sam-checkpoint", str(checkpoint_path),
    "--output-root", str(OUTPUT_ROOT),
    "--device", "cuda",
    "--shard-size", "200",
    "--run-mode", RUN_PHASE,
    "--maximum-new-shards", str(MAXIMUM_NEW_SHARDS),
    "--recovery-output-directory", str(RECOVERY_OUTPUT_ROOT),
]
recovery_pair = find_recovery_pair()
if recovery_pair is not None:
    command.extend(["--recovery-bundle", str(recovery_pair[0]), "--recovery-manifest", str(recovery_pair[1])])
preflight_path = OUTPUT_ROOT / "preflight/model_preflight.json"
preflight_command = [
    sys.executable,
    str(CODE_ROOT / "scripts/run_frozen_development_pipeline.py"),
    "preflight",
    "--cohort", "remainder",
    "--model-input-root", str(REMAINDER_ROOT),
    "--selected-prompt", str(SELECTED_PROMPT),
    "--sam-checkpoint", str(checkpoint_path),
    "--output-path", str(preflight_path),
    "--device", "cuda",
    "--maximum-image-attempts", "5",
]
subprocess.run(preflight_command, check=True, timeout=1800, env=code_environment())
preflight_record = json.loads(preflight_path.read_text(encoding="utf-8"))
if preflight_record["status"] != "passed":
    raise RuntimeError("The bounded model preflight did not pass")
inference_environment = code_environment()
inference_environment["HF_HUB_OFFLINE"] = "1"
inference_environment["TRANSFORMERS_OFFLINE"] = "1"
subprocess.run(command, check=True, env=inference_environment)

screen_root = OUTPUT_ROOT / "toysad8k-development-screen-100-v1"
remainder_root = OUTPUT_ROOT / "toysad8k-development-remainder-600-v1"
screen_record = json.loads((screen_root / "safe/run_record.json").read_text(encoding="utf-8"))
remainder_record = json.loads((remainder_root / "safe/run_record.json").read_text(encoding="utf-8"))
development_progress = json.loads((OUTPUT_ROOT / "development_progress.json").read_text(encoding="utf-8"))
PIPELINE_COMPLETE = development_progress["status"] == "complete"
display(pd.DataFrame([
    {
        "cohort": record["cohort_id"],
        "scenes": record["scene_count"],
        "images": record["image_count"],
        "audit panels": record["audit_panel_count"],
        "status": record["status"],
    }
    for record in (screen_record, remainder_record)
]))
display({
    "model preflight": preflight_record["status"],
    "pipeline complete": PIPELINE_COMPLETE,
    "recovery ZIP": development_progress.get("recovery", {}).get("bundle"),
    "recovery SHA-256": development_progress.get("recovery", {}).get("bundle_sha256"),
    "next action": "finalise the development result" if PIPELINE_COMPLETE else "publish the recovery output privately and run the next incremental version",
})

{"device": "cuda", "dtype": "torch.float16", "event": "preflight_locate_dependencies_ready"}
{"event": "preflight_locate_tokenizer_load_started", "source_kind": "hub"}
{"event": "preflight_locate_tokenizer_ready"}
{"event": "preflight_locate_processor_load_started", "source_kind": "hub"}


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


{"event": "preflight_locate_processor_ready"}
{"event": "preflight_locate_model_weights_load_started", "source_kind": "hub"}


`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:34<00:00, 17.35s/it]
Qwen2ForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Loading checkpoint shards: 100%|██████████| 2/2 [00:0

{"event": "preflight_locate_model_weights_loaded"}
{"device": "cuda", "event": "preflight_locate_model_device_move_started"}
{"device": "cuda", "event": "preflight_locate_model_ready"}


...

{"event": "shard_complete", "shard": 10, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 11, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 12, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 13, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 14, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 15, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 16, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 17, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 18, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 19, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 20, "shards": 60, "stage": "locateanything"}
{"event": "shard_complete", "shard": 21, "shards": 60, "stage": "locateanything"}
{"event": "shard

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()


{"event": "shard_complete", "shard": 10, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 11, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 12, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 13, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 14, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 15, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 16, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 17, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 18, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 19, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 20, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 21, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 22, "shards": 60, "stage": "sam21"}
{"event": "shard_complete", "shard": 23, "shards": 

,cohort,scenes,images,audit panels,status
0,toysad8k-development-screen-100-v1,100,2000,14,complete
1,toysad8k-development-remainder-600-v1,600,12000,15,complete


{'model preflight': 'passed',
 'pipeline complete': True,
 'recovery ZIP': '/kaggle/working/notebook08e_recovery/notebook08e-recovery.zip',
 'recovery SHA-256': '8a68f8a7ba64557250c17a27e4b3cd612b7f8844a847913d9c6eb54320386d26',
 'next action': 'finalise the development result'}

## Backfill the confirmed 300-scene route

The earlier confirmation output predates the 08E prediction bundle. Its LocateAnything and automatic SAM results are reused only after the exact model revision, prompt, proposal policy, configuration, result hash and schema fields verify. DINOv2-small, B0 and P1 are then rebuilt under the current frozen identity. A missing or different field stops the notebook and requires a complete confirmation rerun.

In [6]:
confirmation_root = OUTPUT_ROOT / "toysad8k-development-confirm-300-v1"
confirmation_record = None
CONFIRMATION_COMPLETE = False
if PIPELINE_COMPLETE:
    existing_confirmation_lock = confirmation_root / "model-safe/prediction_lock.json"
    if existing_confirmation_lock.is_file():
        verify_prediction_lock(existing_confirmation_lock)
    else:
        backfill_command = [
            sys.executable,
            str(CODE_ROOT / "scripts/run_frozen_development_pipeline.py"),
            "backfill-ds2",
            "--rgb-root", str(DS2_RGB_ROOT),
            "--mask-route-root", str(DS2_REUSE_ROOT / "mask-routes"),
            "--locate-run-record", str(DS2_REUSE_ROOT / "locateanything_run_record.json"),
            "--automatic-sam-record", str(DS2_REUSE_ROOT / "automatic_sam_run_record.json"),
            "--selected-prompt", str(SELECTED_PROMPT),
            "--output-root", str(OUTPUT_ROOT),
            "--device", "cuda",
            "--shard-size", "200",
            "--run-mode", DS2_BACKFILL_PHASE,
            "--maximum-new-shards", str(MAXIMUM_NEW_SHARDS),
            "--recovery-output-directory", str(DS2_RECOVERY_OUTPUT_ROOT),
        ]
        ds2_recovery_pair = find_recovery_pair(
            "notebook08e-ds2-backfill",
            "VPI_DS2_RESUME_ROOT",
        )
        if ds2_recovery_pair is not None:
            backfill_command.extend(["--recovery-bundle", str(ds2_recovery_pair[0]), "--recovery-manifest", str(ds2_recovery_pair[1])])
        subprocess.run(backfill_command, check=True, env=code_environment())
    confirmation_record = json.loads((confirmation_root / "safe/run_record.json").read_text(encoding="utf-8"))
    CONFIRMATION_COMPLETE = confirmation_record["status"] == "complete"
    display({
        "status": confirmation_record["status"],
        "reused stages": confirmation_record["reused_stages"],
        "rebuilt stages": confirmation_record["rebuilt_stages"],
        "scenes": confirmation_record["scene_count"],
        "images": confirmation_record["image_count"],
        "backfill phase": DS2_BACKFILL_PHASE,
        "final lock": CONFIRMATION_COMPLETE,
    })
else:
    display({"confirmation backfill": "deferred until both GPU cohorts are complete"})

{"event": "shard_complete", "shard": 0, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 1, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 2, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 3, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 4, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 5, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 6, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 7, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 8, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 9, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 10, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 11, "shards": 30, "stage": "dinov2_small"}
{"event": "shard_complete", "shard": 12, "shards":

{'status': 'complete',
 'reused stages': ['locateanything', 'sam21'],
 'rebuilt stages': ['dinov2_small', 'peer_scoring'],
 'scenes': 300,
 'images': 6000,
 'backfill phase': 'full',
 'final lock': True}

## Terminal outcomes and coverage

In [7]:
stage_rows = []
available_records = [screen_record, remainder_record]
if confirmation_record is not None:
    available_records.append(confirmation_record)
for record in available_records:
    for stage_name, stage in record["stages"].items():
        stage_rows.append({
            "cohort": record["cohort_id"],
            "stage": stage_name,
            "inputs": stage["input_count"],
            "processed this run": stage["processed_this_run"],
            "resumed": stage["resumed"],
            "terminal failures": stage["terminal_failures"],
            "zero proposal": stage["failure_counts"]["zero_proposal"],
            "empty mask": stage["failure_counts"]["empty_mask"],
            "feature failure": stage["failure_counts"]["feature_failure"],
        })
stage_table = pd.DataFrame(stage_rows)
display(stage_table)
stage_table.to_csv(OUTPUT_ROOT / "stage_coverage.csv", index=False)

,cohort,stage,inputs,processed this run,resumed,terminal failures,zero proposal,empty mask,feature failure
0,toysad8k-development-screen-100-v1,dinov2_small,2000,0,2000,17,17,0,0
1,toysad8k-development-screen-100-v1,locateanything,2000,0,2000,17,17,0,0
2,toysad8k-development-screen-100-v1,peer_scoring,2000,0,2000,17,17,0,0
3,toysad8k-development-screen-100-v1,sam21,2000,0,2000,17,17,0,0
4,toysad8k-development-remainder-600-v1,dinov2_small,12000,10000,2000,71,69,0,0
5,toysad8k-development-remainder-600-v1,locateanything,12000,10000,2000,69,69,0,0
6,toysad8k-development-remainder-600-v1,peer_scoring,12000,12000,0,69,69,0,0
7,toysad8k-development-remainder-600-v1,sam21,12000,10000,2000,69,69,0,0
8,toysad8k-development-confirm-300-v1,dinov2_small,6000,6000,0,38,38,0,0
9,toysad8k-development-confirm-300-v1,peer_scoring,6000,6000,0,38,38,0,0


## Locked model-safe artifacts

In [8]:
lock_rows = []
if PIPELINE_COMPLETE and CONFIRMATION_COMPLETE:
    for root in (screen_root, remainder_root, confirmation_root):
        lock_path = root / "model-safe/prediction_lock.json"
        lock = verify_prediction_lock(lock_path)
        lock_rows.append({
            "cohort": lock["cohort_id"],
            "scenes": lock["scene_count"],
            "images": lock["image_count"],
            "artifacts": lock["artifact_count"],
            "identity SHA-256": lock["identity_sha256"],
            "lock SHA-256": file_sha256(lock_path),
        })
lock_table = pd.DataFrame(lock_rows)
display(lock_table)
lock_table.to_csv(OUTPUT_ROOT / "prediction_locks.csv", index=False)

,cohort,scenes,images,artifacts,identity SHA-256,lock SHA-256
0,toysad8k-development-screen-100-v1,100,2000,3,c7c4c8df023c30884baf0fd2693fdbaece5840f1fb5e8d...,d035bfb59af2e4a296361283ee5f263d5ad2a8682efa93...
1,toysad8k-development-remainder-600-v1,600,12000,3,c7c4c8df023c30884baf0fd2693fdbaece5840f1fb5e8d...,f5c88d87836c2b1a82c2499edf71af753fb125a74e3ea1...
2,toysad8k-development-confirm-300-v1,300,6000,3,c7c4c8df023c30884baf0fd2693fdbaece5840f1fb5e8d...,7d7b683e3edfe2c8ebd2b6623d16e292870cdb70bed845...


## Prediction-only qualitative evidence

In [9]:
figure_paths = []
figure_roots = [screen_root, remainder_root]
if CONFIRMATION_COMPLETE:
    figure_roots.append(confirmation_root)
for root in figure_roots:
    if not (root / "review-only/audit_evidence.json").is_file():
        continue
    evidence = json.loads((root / "review-only/audit_evidence.json").read_text(encoding="utf-8"))
    for item in evidence["items"]:
        figure_paths.extend((root / item["prediction_panel"], root / item["sam_evidence_panel"]))

preview_paths = []
for root in figure_roots:
    preview_paths.extend([path for path in figure_paths if path.is_relative_to(root)][:4])
for path in preview_paths:
    display(NotebookImage(filename=str(path)))
display({"prediction-only audit figures": len(figure_paths)})

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

...

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

{'prediction-only audit figures': 90}

## Verify and combine the full development role

In [10]:
combined_path = OUTPUT_ROOT / "full-development-lock.json"
development_record = None
if PIPELINE_COMPLETE and CONFIRMATION_COMPLETE:
    subprocess.run([
        sys.executable,
        str(CODE_ROOT / "scripts/run_frozen_development_pipeline.py"),
        "combine",
        "--ds1-lock", str(screen_root / "model-safe/prediction_lock.json"),
        "--remainder-lock", str(remainder_root / "model-safe/prediction_lock.json"),
        "--ds2-lock", str(confirmation_root / "model-safe/prediction_lock.json"),
        "--output-path", str(combined_path),
    ], check=True, env=code_environment())
    development_record = json.loads(combined_path.read_text(encoding="utf-8"))
    display({
        "status": development_record["status"],
        "development scenes represented": development_record["scene_count"],
        "confirmation reused": development_record.get("ds2_reused_only_after_full_identity_check", False),
    })
else:
    display({"combined development lock": "not published before exact full coverage"})

{"cohort_id": "toysad8k-development-full-1000-v1", "component_locks": [{"cohort_id": "toysad8k-development-screen-100-v1", "lock_sha256": "d035bfb59af2e4a296361283ee5f263d5ad2a8682efa93f70b5e9707f0b00404"}, {"cohort_id": "toysad8k-development-remainder-600-v1", "lock_sha256": "f5c88d87836c2b1a82c2499edf71af753fb125a74e3ea1354d1bb077fa8d337f"}, {"cohort_id": "toysad8k-development-confirm-300-v1", "lock_sha256": "7d7b683e3edfe2c8ebd2b6623d16e292870cdb70bed84586a14aae950739d7bb"}], "ds2_reused_only_after_full_identity_check": true, "identity": {"configuration_sha256": "6af6710e2b4cf6908ff2ac2a567a9fec848b8a9414b1f8fe0e9dfe3a6c8a090a", "dinov2_sha256": "35a7fb09a5edf3f93ddd221029bfcda351e860e1797f5cb8efe2f718219aed91", "locateanything_sha256": "dac3f379588afca42d1070ad8c1010975e1e231b323e48a782f49ad66b7ca4af", "preprocessing_sha256": "31a77a0b4e6acd5801502e8acd2852c4f84ab8577a4f4cc65d0208947300c14c", "sam21_sha256": "39690d431d70c2192a6ba0c4530785c9208e07b035842d05022a67fb700a850b", "schem

{'status': 'verified',
 'development scenes represented': 1000,
 'confirmation reused': True}

## Reference-only development analysis

The prediction locks above are the boundary between inference and evaluation. Any later use of the private development correspondence may measure alignment or compare multi-view fusion methods, but it cannot support a deployment-valid claim because a web upload does not provide evaluator correspondence. Notebook 08F performs that analysis without altering these predictions.

## Aggregate ClearML record

In [11]:
tracking_rows = []
if PIPELINE_COMPLETE and CONFIRMATION_COMPLETE:
    for root, task_name in (
        (screen_root, "frozen-development-screen-100"),
        (remainder_root, "frozen-development-remainder-600"),
        (confirmation_root, "frozen-development-confirmation-300"),
    ):
        receipt = root / "tracking/clearml_receipt.json"
        tracking = finish_clearml(root / "safe", task_name, receipt, "final")
        tracking_rows.append({
            "cohort": root.name,
            "status": tracking["status"],
            "task_id": tracking.get("task_id"),
            "record kind": "final",
        })
else:
    recovery_safe = OUTPUT_ROOT / "tracking-safe"
    receipt = OUTPUT_ROOT / "tracking/clearml_recovery_receipt.json"
    tracking = finish_clearml(
        recovery_safe,
        "frozen-development-recovery",
        receipt,
        "recovery",
    )
    tracking_rows.append({
        "cohort": "development checkpoint",
        "status": tracking["status"],
        "task_id": tracking.get("task_id"),
        "record kind": "recovery",
    })
display(pd.DataFrame(tracking_rows))

Traceback (most recent call last):
  File "/kaggle/input/datasets/esreekarreddy/vpi-code-snapshot/scripts/finalize_ds2_clearml.py", line 300, in <module>
    main()
  File "/kaggle/input/datasets/esreekarreddy/vpi-code-snapshot/scripts/finalize_ds2_clearml.py", line 251, in main
    record = validate_safe_root(root, expected_stage=args.stage)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/input/datasets/esreekarreddy/vpi-code-snapshot/scripts/finalize_ds2_clearml.py", line 149, in validate_safe_root
    raise last_error
  File "/kaggle/input/datasets/esreekarreddy/vpi-code-snapshot/scripts/finalize_ds2_clearml.py", line 140, in validate_safe_root
    record, _manifest = validate_tracking_safe_root(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/input/datasets/esreekarreddy/vpi-code-snapshot/src/visual_peer_inspector/tracking_safe_artifacts.py", line 62, in validate_tracking_safe_root
    raise ValueError("ClearML safe root differs

,cohort,status,task_id,record kind
0,toysad8k-development-screen-100-v1,disabled,None,final
1,toysad8k-development-remainder-600-v1,disabled,None,final
2,toysad8k-development-confirm-300-v1,failed,None,final


## Final artifact register

In [12]:
artifact_rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and not path.name.endswith(".incomplete"):
        artifact_rows.append({
            "path": path.relative_to(OUTPUT_ROOT).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        })
artifact_register = pd.DataFrame(artifact_rows)
artifact_register.to_csv(OUTPUT_ROOT / "artifact_register.csv", index=False)
display(artifact_register)

,path,bytes,sha256
0,development_progress.json,11915,97c45a2a7f4028680ff61c7075306740d6e638fb93f672...
1,full-development-lock.json,1459,9d8505d2bccdcf9d6e59ebf39c52676278517ff6564c39...
2,prediction_locks.csv,594,e571c9931a43cdbd9b2622a182c16e50a0283773df9bbf...
3,preflight/model_preflight.json,436,ea7ab62f8a8d1edd2b4ca3b2c2bd791101b047da578171...
4,stage_coverage.csv,827,5c37e2aa55a9577905506c67347eb2badcda5e5534a20e...
...,...,...,...
1002,toysad8k-development-screen-100-v1/sam21/shard...,691073,0b6fd06a1cddb2c5b146873d45ba04327814f97901d622...
1003,toysad8k-development-screen-100-v1/sam21/shard...,460,b51dc94774679085e4bd3f4171995c2a0ff2cebc31214c...
1004,toysad8k-development-screen-100-v1/tracking/cl...,637,41d12bb9febf7f793145a8a169a1cf631329e037da44bb...
1005,tracking-safe/artifact_manifest.json,174,c19a42d45ef6a68c2115e578763bcbcdb10572d7955752...


## Result boundary

The 100-scene screen and 600-scene remainder are complete when their prediction locks verify. The 300-scene DS2 DINOv2-small and P1 backfill may also use bounded recovery versions and cannot publish a lock before all 6,000 views are covered. The full 1,000-scene development result is complete only when all three locks pass the same identity contract. Calibration and internal-lock results are not part of this notebook.